In [1]:
!pip install unsloth
!pip install IProgress 
!pip install trl

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 141.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 238.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 314.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 275.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 261.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 268.7 MB/s  0:00:03eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 276.6 MB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 290.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/

In [8]:
!pip install transformers
!pip uninstall -y unsloth

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: unsloth 2025.12.9
Uninstalling unsloth-2025.12.9:
  Successfully uninstalled unsloth-2025.12.9


In [4]:
from pathlib import Path

with open("data.jsonl","w", encoding="utf-8") as out:
    for path in Path("data").rglob("*"):
        if path.name == "data.json":
            with path.open("r", encoding="utf-8") as f:
                obj = json.load(f)
                out.write(json.dumps(obj, ensure_ascii=False) + "\n")

        elif path.name == "data.jsonl":
            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        out.write(line.rstrip("\n") + "\n")

In [21]:
import json
data = []
with open('data.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

data = [{"prompt": item["inputs"], "completion": item["outputs"]} for item in data]
    
with open('data.jsonl', 'w', encoding='utf-8') as f:
     for item in data:
         f.write(json.dumps(item) + '\n')

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model

model_name = "google/long-t5-tglobal-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")

# LoRA config
lora_config = LoraConfig(
    r=16,
    target_modules=[
        "SelfAttention.q",
        "SelfAttention.k",
        "SelfAttention.v",
        "SelfAttention.o",
        "DenseReluDense.wi",
        "DenseReluDense.wo"
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)

model.tie_weights()
model.print_trainable_parameters()

The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


trainable params: 2,260,992 || all params: 299,197,056 || trainable%: 0.7557


In [22]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="data.jsonl", split="train")

Generating train split: 727 examples [00:00, 49721.31 examples/s]


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="long-t5-tglobal-finetuned",
        per_device_train_batch_size=4,   # small batch size for Colab GPU
        gradient_accumulation_steps=4,   # accumulate gradients to simulate larger batch
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        logging_steps=50,
        save_strategy="epoch"
    )
)

trainer.train()
model.save_pretrained("long-t5-tglobal-finetuned")

The model is already on multiple devices. Skipping the move to device specified in `args`.
/usr/local/lib/python3.10/dist-packages/apex/_autocast_utils.py:26: FutureWarning: `torch.cuda.amp.autocast_mode._cast(value, dtype)` is deprecated. Please use `torch.amp.autocast_mode._cast(value, 'cuda', dtype)` instead.
  return torch.cuda.amp.autocast_mode._cast(args, torch.get_autocast_gpu_dtype())
/usr/local/lib/python3.10/dist-packages/apex/normalization/fused_layer_norm.py:214: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


Step,Training Loss
50,1.327900


/usr/local/lib/python3.10/dist-packages/apex/_autocast_utils.py:26: FutureWarning: `torch.cuda.amp.autocast_mode._cast(value, dtype)` is deprecated. Please use `torch.amp.autocast_mode._cast(value, 'cuda', dtype)` instead.
  return torch.cuda.amp.autocast_mode._cast(args, torch.get_autocast_gpu_dtype())
/usr/local/lib/python3.10/dist-packages/apex/normalization/fused_layer_norm.py:214: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
